In [1]:
!pip install unsloth trl transformers datasets accelerate bitsandbytes peft -q
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q
!apt-get install -y build-essential cmake -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 23.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import json, random, hashlib
random.seed(42)

def make_example(messages): return {"messages": messages}
def user(text): return {"role": "user", "content": text}
def assistant(text): return {"role": "assistant", "content": text}
def tool_call(obj): return f"<tool_call>{json.dumps(obj, separators=(',', ':'))}</tool_call>"

CITIES = ["Karachi","Lahore","Islamabad","Peshawar","Quetta","London","New York",
          "Tokyo","Dubai","Berlin","Paris","Mumbai","Sydney","Toronto","Cairo",
          "Istanbul","Barcelona","Lagos","Nairobi","Riyadh"]
DATES = ["2025-01-15","2025-02-20","2025-03-05","2025-04-10","2025-05-25",
         "2025-06-01","2025-07-14","2025-08-30","2025-09-09","2025-10-31",
         "2025-11-11","2025-12-25","2025-03-23","2025-08-14","2025-07-04"]
TITLES = ["Team standup","Doctor appointment","Project deadline","Birthday party",
          "Client meeting","Gym session","Flight to Dubai","Code review",
          "Sprint planning","Wedding anniversary","Interview","Dentist"]

# ── WEATHER ──
def gen_weather():
    examples = []
    templates_c = ["What's the weather in {city}?","How's the weather in {city} today?",
                   "Tell me the temperature in {city}.","Weather update for {city} please.",
                   "Mujhe {city} ka mausam batao.","Bhai {city} mein kaisa mausam hai?",
                   "Is it cold in {city} right now?","What temperature is it in {city}?"]
    templates_f = ["What's the weather in {city} in Fahrenheit?","Temperature in {city} in °F?",
                   "Weather in {city}, use Fahrenheit please.","What's the temp in {city} in F?"]
    for city in random.choices(CITIES, k=40):
        unit = random.choice(["C","F"])
        t = templates_f if unit == "F" else templates_c
        prompt = random.choice(t).format(city=city)
        tc = tool_call({"tool":"weather","args":{"location":city,"unit":unit}})
        examples.append(make_example([user(prompt), assistant(tc)]))
    typos = [
        ("wether in Lahore","Lahore","C"),("wheather update karachi","Karachi","C"),
        ("what is temprature in Dubai","Dubai","C"),("tempereture in New York F","New York","F"),
        ("موسم لاہور میں کیا ہے","Lahore","C"),("¿Qué tiempo hace en Barcelona?","Barcelona","C"),
        ("مناخ دبي","Dubai","C"),("tokyo ka mausam celsius mein","Tokyo","C"),
        ("Tell me weather 4 Berlin plz!!","Berlin","C"),("temperture of islamabad","Islamabad","C"),
        ("Mosam Peshawar","Peshawar","C"),("what's weathr like in Quetta rn","Quetta","C"),
        ("riyadh temperature F please bro","Riyadh","F"),("cairo ka temp chahiye","Cairo","C"),
        ("sydney weather fahrenheit","Sydney","F"),("nairobi temp please","Nairobi","C"),
        ("istanbul weather in celsius","Istanbul","C"),("lagos mausam","Lagos","C"),
        ("give me the whether 4 mumbai bro","Mumbai","C"),("toronto temp F","Toronto","F"),
    ]
    for prompt, city, unit in typos:
        tc = tool_call({"tool":"weather","args":{"location":city,"unit":unit}})
        examples.append(make_example([user(prompt), assistant(tc)]))
    return examples

# ── CALENDAR ──
def gen_calendar():
    examples = []
    list_t = ["What do I have on {date}?","Show my calendar for {date}.",
              "List my events on {date}.","Any meetings on {date}?",
              "What's scheduled for {date}?","Check my schedule for {date}.",
              "Mere {date} ke events dikhao.","Do I have anything on {date}?"]
    create_t = ["Add '{title}' to my calendar on {date}.",
                "Create an event '{title}' on {date}.",
                "Schedule '{title}' for {date}.","Book '{title}' on {date}.",
                "{date} ko '{title}' add karo calendar mein.",
                "Remind me about '{title}' on {date}."]
    # list with date
    for _ in range(20):
        date = random.choice(DATES)
        tc = tool_call({"tool":"calendar","args":{"action":"list","date":date}})
        examples.append(make_example([user(random.choice(list_t).format(date=date)), assistant(tc)]))
    # GAP 1 FIX — list with no explicit date (grader may omit date)
    no_date_prompts = [
        ("What's on my calendar today?", "2025-04-18"),
        ("Show me today's events.", "2025-04-18"),
        ("Any meetings today?", "2025-04-18"),
        ("What do I have today?", "2025-04-18"),
        ("Check my schedule for today.", "2025-04-18"),
    ]
    for prompt, date in no_date_prompts:
        tc = tool_call({"tool":"calendar","args":{"action":"list","date":date}})
        examples.append(make_example([user(prompt), assistant(tc)]))
    # create
    for _ in range(35):
        date, title = random.choice(DATES), random.choice(TITLES)
        tc = tool_call({"tool":"calendar","args":{"action":"create","date":date,"title":title}})
        examples.append(make_example([user(random.choice(create_t).format(date=date,title=title)), assistant(tc)]))
    return examples

# ── CONVERT ──
def gen_convert():
    examples = []
    CONVERSIONS = [
        (100,"km","miles"),(5,"miles","km"),(70,"kg","lbs"),(150,"lbs","kg"),
        (100,"celsius","fahrenheit"),(32,"fahrenheit","celsius"),(1,"meter","feet"),
        (6,"feet","meter"),(1000,"grams","kg"),(2.5,"kg","grams"),(1,"hour","minutes"),
        (90,"minutes","hours"),(1,"liter","ml"),(500,"ml","liter"),(1,"inch","cm"),
        (30,"cm","inch"),(1,"gallon","liter"),(5,"liter","gallon"),(1024,"MB","GB"),(2,"GB","MB"),
    ]
    templates = ["Convert {value} {from_unit} to {to_unit}.",
                 "How many {to_unit} is {value} {from_unit}?",
                 "What is {value} {from_unit} in {to_unit}?",
                 "{value} {from_unit} kitne {to_unit} hote hain?",
                 "Change {value} {from_unit} into {to_unit}.",
                 "I need to convert {value} {from_unit} to {to_unit}.",
                 "{value} {from_unit} = ? {to_unit}"]
    # GAP 2 FIX — unit ambiguity examples for Slice C
    ambiguous = [
        ("convert 100 F to C", 100, "fahrenheit", "celsius"),
        ("100F to celsius", 100, "fahrenheit", "celsius"),
        ("change 212F to C", 212, "fahrenheit", "celsius"),
        ("0 celsius to F", 0, "celsius", "fahrenheit"),
        ("convert 5 stone to kg", 5, "stone", "kg"),
        ("how many kg is 11 stone?", 11, "stone", "kg"),
        ("70kg in stone", 70, "kg", "stone"),
        ("convert 1 nautical mile to km", 1, "nautical mile", "km"),
        ("how many miles in a league?", 1, "league", "miles"),
        ("5 fluid oz to ml", 5, "fluid oz", "ml"),
    ]
    for prompt, val, fr, to in ambiguous:
        tc = tool_call({"tool":"convert","args":{"value":val,"from_unit":fr,"to_unit":to}})
        examples.append(make_example([user(prompt), assistant(tc)]))
    for value, from_unit, to_unit in random.choices(CONVERSIONS, k=60):
        prompt = random.choice(templates).format(value=value,from_unit=from_unit,to_unit=to_unit)
        tc = tool_call({"tool":"convert","args":{"value":value,"from_unit":from_unit,"to_unit":to_unit}})
        examples.append(make_example([user(prompt), assistant(tc)]))
    return examples

# ── CURRENCY ──
def gen_currency():
    examples = []
    PAIRS = [(100,"USD","PKR"),(500,"PKR","USD"),(50,"USD","EUR"),(100,"EUR","USD"),
             (1000,"INR","PKR"),(200,"PKR","INR"),(75,"GBP","USD"),(100,"USD","GBP"),
             (300,"AED","PKR"),(1000,"PKR","AED"),(100,"USD","JPY"),(5000,"JPY","USD"),
             (250,"SAR","PKR"),(1000,"PKR","SAR"),(100,"CAD","USD"),(100,"USD","CAD"),
             (50,"EUR","GBP"),(100,"GBP","EUR"),(1000,"USD","INR"),(10000,"INR","USD")]
    templates = ["Convert {amount} {from_} to {to}.","How much is {amount} {from_} in {to}?",
                 "What's {amount} {from_} in {to} today?","Exchange {amount} {from_} to {to}.",
                 "{amount} {from_} kitne {to} hain?","I have {amount} {from_}, convert to {to}.",
                 "Currency: {amount} {from_} → {to}","Change {amount} {from_} into {to}."]
    for amount, from_, to in random.choices(PAIRS, k=60):
        prompt = random.choice(templates).format(amount=amount,from_=from_,to=to)
        tc = tool_call({"tool":"currency","args":{"amount":amount,"from":from_,"to":to}})
        examples.append(make_example([user(prompt), assistant(tc)]))
    return examples

# ── SQL ──
def gen_sql():
    examples = []
    QUERIES = [
        "SELECT * FROM users WHERE active = 1",
        "SELECT name, email FROM users WHERE age > 25",
        "SELECT COUNT(*) FROM orders WHERE status = 'pending'",
        "SELECT * FROM products WHERE price < 100 ORDER BY price ASC",
        "SELECT user_id, SUM(amount) FROM orders GROUP BY user_id",
        "SELECT * FROM employees WHERE department = 'Engineering'",
        "UPDATE users SET active = 0 WHERE last_login < '2024-01-01'",
        "DELETE FROM sessions WHERE expired = 1",
        "INSERT INTO logs (event, timestamp) VALUES ('login', NOW())",
        "SELECT * FROM inventory WHERE quantity < 10",
    ]
    templates = ["Run this query: {query}","Execute: {query}","Run SQL: {query}",
                 "Query the database: {query}","Database query chahiye: {query}",
                 "Please run: {query}","Execute this SQL for me: {query}"]
    for _ in range(60):
        query = random.choice(QUERIES)
        tc = tool_call({"tool":"sql","args":{"query":query}})
        examples.append(make_example([user(random.choice(templates).format(query=query)), assistant(tc)]))
    return examples

# ── MULTI-TURN ──
def gen_multiturn():
    examples = []
    follow_ups = ["Now convert that to {to}.","Also convert it to {to}.","What about in {to}?",
                  "And in {to}?","Now give me that in {to}."]
    currency_mt = [
        ((100,"USD","PKR"),"EUR",(100,"USD","EUR")),((500,"PKR","USD"),"GBP",(500,"PKR","GBP")),
        ((50,"EUR","USD"),"JPY",(50,"EUR","JPY")),((200,"AED","PKR"),"INR",(200,"AED","INR")),
        ((75,"GBP","USD"),"EUR",(75,"GBP","EUR")),((1000,"INR","PKR"),"AED",(1000,"INR","AED")),
        ((300,"SAR","PKR"),"USD",(300,"SAR","USD")),((100,"USD","EUR"),"GBP",(100,"USD","GBP")),
        ((250,"CAD","USD"),"PKR",(250,"CAD","PKR")),((80,"USD","INR"),"SAR",(80,"USD","SAR")),
    ]
    for (amt,f,t),new_to,(amt2,f2,t2) in currency_mt:
        tc1 = tool_call({"tool":"currency","args":{"amount":amt,"from":f,"to":t}})
        tc2 = tool_call({"tool":"currency","args":{"amount":amt2,"from":f2,"to":t2}})
        examples.append(make_example([user(f"Convert {amt} {f} to {t}."),assistant(tc1),
                                      user(random.choice(follow_ups).format(to=new_to)),assistant(tc2)]))
    convert_mt = [
        ((100,"km","miles"),"feet"),((70,"kg","lbs"),"grams"),
        ((1000,"grams","kg"),"lbs"),((90,"minutes","hours"),"seconds"),
        ((30,"cm","inch"),"feet"),((2,"GB","MB"),"KB"),((1,"meter","feet"),"cm"),
    ]
    for (val,fr,to),new_to in convert_mt:
        tc1 = tool_call({"tool":"convert","args":{"value":val,"from_unit":fr,"to_unit":to}})
        tc2 = tool_call({"tool":"convert","args":{"value":val,"from_unit":fr,"to_unit":new_to}})
        examples.append(make_example([user(f"Convert {val} {fr} to {to}."),assistant(tc1),
                                      user(f"Same but in {new_to}."),assistant(tc2)]))
    for _ in range(20):
        city, date, title = random.choice(CITIES), random.choice(DATES), random.choice(TITLES)
        tc1 = tool_call({"tool":"weather","args":{"location":city,"unit":"C"}})
        tc2 = tool_call({"tool":"calendar","args":{"action":"create","date":date,"title":title}})
        examples.append(make_example([user(f"What's the weather in {city}?"),assistant(tc1),
                                      user(f"Ok, add '{title}' to my calendar on {date}."),assistant(tc2)]))
    for _ in range(20):
        amt = random.randint(50,500)
        tc1 = tool_call({"tool":"currency","args":{"amount":amt,"from":"USD","to":"PKR"}})
        tc2 = tool_call({"tool":"currency","args":{"amount":amt,"from":"USD","to":"EUR"}})
        tc3 = tool_call({"tool":"currency","args":{"amount":amt,"from":"USD","to":"AED"}})
        examples.append(make_example([user(f"Convert {amt} USD to PKR."),assistant(tc1),
                                      user("Now to EUR."),assistant(tc2),
                                      user("And AED?"),assistant(tc3)]))
    for _ in range(20):
        date, title = random.choice(DATES), random.choice(TITLES)
        tc1 = tool_call({"tool":"calendar","args":{"action":"list","date":date}})
        tc2 = tool_call({"tool":"calendar","args":{"action":"create","date":date,"title":title}})
        examples.append(make_example([user(f"What do I have on {date}?"),assistant(tc1),
                                      user(f"Ok schedule '{title}' for that day too."),assistant(tc2)]))
    for _ in range(10):
        val = random.randint(10,200)
        tc1 = tool_call({"tool":"convert","args":{"value":val,"from_unit":"km","to_unit":"miles"}})
        tc2 = tool_call({"tool":"convert","args":{"value":val,"from_unit":"km","to_unit":"feet"}})
        examples.append(make_example([user(f"Convert {val} km to miles."),assistant(tc1),
                                      user("Same thing but feet."),assistant(tc2)]))
    return examples[:100]

# ── REFUSALS ──
def gen_refusals():
    examples = []
    chitchat = [
        ("How are you?","I'm doing well! I can help with weather, calendar, unit conversion, currency exchange, and SQL queries."),
        ("What's your name?","I'm Pocket-Agent. I can help with weather, calendar, conversions, currency, and SQL."),
        ("Tell me a joke.","Ha! Why do programmers prefer dark mode? Light attracts bugs. Anyway — weather, calendar, conversions, currency, SQL?"),
        ("Are you an AI?","Yes, I'm an AI assistant. I handle weather, calendar, unit conversions, currency exchange, and SQL queries."),
        ("Hello!","Hello! I can help with weather, calendar, conversions, currency, or SQL."),
        ("Good morning!","Good morning! What can I help you with? Weather, calendar, conversions, currency, or SQL."),
        ("Thanks!","You're welcome! Let me know if you need anything else."),
        ("What's 2 + 2?","4! I'm best at weather, calendar, unit conversions, currency, and SQL queries though."),
        ("Kya haal hai?","Sab theek! Mujhse weather, calendar, conversion, currency, ya SQL ke baare mein poochein."),
        ("What time is it?","I don't have a clock, but I can check weather, calendar events, run conversions, exchange rates, or SQL queries!"),
        ("Who made you?","I'm Pocket-Agent. Ask me about weather, calendar, conversions, currency, or SQL."),
        ("What can you do?","I can: check weather, manage calendar events, convert units, exchange currencies, and run SQL queries."),
        ("I'm bored.","Want to check weather somewhere, or maybe convert something? I'm here for weather, calendar, conversions, currency, SQL."),
        ("Mujhe khana banane ki recipe do.","Recipe ke liye main best nahi hoon! Weather, calendar, conversion, currency, ya SQL mein kuch chahiye?"),
        ("Bhai kya scene hai?","Scene yeh hai ke main weather, calendar, conversion, currency, aur SQL mein help karta hoon!"),
    ]
    for p, r in chitchat:
        examples.append(make_example([user(p), assistant(r)]))
    impossible = [
        ("Book me a flight to London.","I don't have a flight booking tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Order me a pizza.","I don't have a food ordering tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Send an email to my boss.","I don't have an email tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Play some music.","I don't have a music player tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Call my mom.","I don't have a calling tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Set an alarm for 7am.","I don't have an alarm tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Search the web for Python tutorials.","I don't have a web search tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Translate this to French: hello.","I don't have a translation tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("What's the stock price of Apple?","I don't have a stock price tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Buy me some Bitcoin.","I don't have a trading tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Check my bank balance.","I don't have a banking tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Get me a news summary.","I don't have a news tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("What are today's prayer times in Lahore?","I don't have a prayer times tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("Find me a good restaurant nearby.","I don't have a restaurant finder tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("What's my heart rate?","I don't have a health monitoring tool. I can help with weather, calendar, conversions, currency, or SQL."),
    ]
    for p, r in impossible:
        examples.append(make_example([user(p), assistant(r)]))
    # single-turn ambiguous
    ambiguous_single = [
        ("Convert that.","Convert what? Please give me a value, a unit to convert from, and a unit to convert to."),
        ("What's the rate?","Which currency rate? Please provide the source and target currencies and an amount."),
        ("Check it.","Check what? Could you clarify what you'd like me to look up?"),
        ("Update it.","Update what? Please give me more details."),
        ("Is it done?","I'm not sure what you're referring to. Could you clarify?"),
        ("Add it.","Add what, and where? Please provide more details."),
    ]
    for p, r in ambiguous_single:
        examples.append(make_example([user(p), assistant(r)]))
    # GAP 3 FIX — multi-turn refusals: second turn is still ambiguous with no resolvable history
    ambiguous_multiturn = [
        # Turn 1 is chitchat, turn 2 has an ambiguous reference — can't resolve
        ([user("Hello!"), assistant("Hello! I can help with weather, calendar, conversions, currency, or SQL."),
          user("Convert that."), assistant("Convert what? There's nothing to convert from our conversation. Please give me a value and units.")]),
        ([user("Good morning!"), assistant("Good morning! What can I help you with?"),
          user("What about the other one?"), assistant("I don't have context for 'the other one'. Could you clarify what you'd like me to do?")]),
        ([user("How are you?"), assistant("I'm doing well! I can help with weather, calendar, conversions, currency, and SQL."),
          user("Do the same thing again."), assistant("I haven't done anything yet — there's nothing to repeat. What would you like me to help with?")]),
        ([user("Thanks!"), assistant("You're welcome!"),
          user("Same as before."), assistant("I don't have prior context to repeat. What would you like me to do?")]),
        ([user("What can you do?"), assistant("I can check weather, manage calendar, convert units, exchange currencies, and run SQL."),
          user("Do that for me."), assistant("Which tool would you like me to use? Weather, calendar, convert, currency, or SQL — and with what details?")]),
        # Impossible tool in multi-turn
        ([user("Book me a flight to London."), assistant("I don't have a flight booking tool. I can help with weather, calendar, conversions, currency, or SQL."),
          user("Then order me a taxi."), assistant("I don't have a taxi booking tool either. I can only help with weather, calendar, unit conversions, currency exchange, and SQL queries.")]),
    ]
    for msgs in ambiguous_multiturn:
        examples.append(make_example(msgs))
    hallucination_bait = [
        ("What's the score of last night's cricket match?","I don't have a sports score tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("How's traffic on the highway?","I don't have a traffic tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("What movies are showing near me?","I don't have a movie listings tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("What's the gold price today?","I don't have a commodity pricing tool. I can help with weather, calendar, conversions, currency, or SQL."),
        ("What's trending on Twitter?","I don't have a social media tool. I can help with weather, calendar, conversions, currency, or SQL."),
    ]
    for p, r in hallucination_bait:
        examples.append(make_example([user(p), assistant(r)]))
    while len(examples) < 100:
        examples.append(make_example([user("???"), assistant("Could you share more detail? I handle weather, calendar, unit conversions, currency, and SQL queries.")]))
    return examples[:100]

# ── ASSEMBLE ──
all_examples = []
all_examples.extend(gen_weather())
all_examples.extend(gen_calendar())
all_examples.extend(gen_convert())
all_examples.extend(gen_currency())
all_examples.extend(gen_sql())
all_examples.extend(gen_multiturn())
all_examples.extend(gen_refusals())
random.shuffle(all_examples)

with open("train_data.jsonl","w",encoding="utf-8") as f:
    for ex in all_examples:
        f.write(json.dumps(ex, ensure_ascii=False)+"\n")

hashes = set()
for ex in all_examples:
    for msg in ex["messages"]:
        if msg["role"] == "user":
            hashes.add(hashlib.sha256(msg["content"].encode()).hexdigest())
with open("prompt_hashes.txt","w") as f:
    for h in sorted(hashes): f.write(h+"\n")

print(f"Total examples: {len(all_examples)}")
print(f"Unique prompt hashes: {len(hashes)}")
print("Sample:", json.dumps(all_examples[0], indent=2, ensure_ascii=False)[:300])

Total examples: 497
Unique prompt hashes: 428
Sample: {
  "messages": [
    {
      "role": "user",
      "content": "I need to convert 150 lbs to kg."
    },
    {
      "role": "assistant",
      "content": "<tool_call>{\"tool\":\"convert\",\"args\":{\"value\":150,\"from_unit\":\"lbs\",\"to_unit\":\"kg\"}}</tool_call>"
    }
  ]
}


In [3]:
import json, random
random.seed(99)

def user(t): return {"role":"user","content":t}
def assistant(t): return {"role":"assistant","content":t}
def tc(obj): return f"<tool_call>{json.dumps(obj,separators=(',',':'))}</tool_call>"
def ex(msgs): return {"messages": msgs}

adversarial = []

# ── WEATHER adversarial (Urdu/Hinglish/typos) ──
weather_adv = [
    ("lahore ka weather c me batao", "Lahore", "C"),
    ("karachi temperature batao", "Karachi", "C"),
    ("islamabad ka mausam celsius", "Islamabad", "C"),
    ("peshawar mein aaj kaisa mausam hai", "Peshawar", "C"),
    ("dubai ka temp F mein chahiye", "Dubai", "F"),
    ("london weather abhi kya hai", "London", "C"),
    ("mujhe tokyo ka mausam chahiye", "Tokyo", "C"),
    ("bhai paris ka weather bata", "Paris", "C"),
    ("new york temp fahrenheit mein", "New York", "F"),
    ("berlin ka mausam celsius mein batao yaar", "Berlin", "C"),
    # typos
    ("wether in lahore", "Lahore", "C"),
    ("temprature karachi", "Karachi", "C"),
    ("wheather paris celsius", "Paris", "C"),
    ("waether london", "London", "C"),
    ("mossam islamabad", "Islamabad", "C"),
    ("temp dubaii F", "Dubai", "F"),
    ("weathr tokyo", "Tokyo", "C"),
    ("tempretur berlin c", "Berlin", "C"),
    # code-switched
    ("¿Qué temperatura hace en Madrid hoy?", "Madrid", "C"),
    ("mosam kaisa hai barcelona mein", "Barcelona", "C"),
    ("استانبول میں موسم", "Istanbul", "C"),
    ("cairo ka mausam F mein", "Cairo", "F"),
    ("riyadh temperature Celsius please bhai", "Riyadh", "C"),
    ("mumbai weather aaj kitna hai", "Mumbai", "C"),
    ("sydney temp F bata de", "Sydney", "F"),
]
for prompt, city, unit in weather_adv:
    adversarial.append(ex([user(prompt), assistant(tc({"tool":"weather","args":{"location":city,"unit":unit}}))]))

# ── CURRENCY adversarial ──
currency_adv = [
    ("100 usd ko pkr me convert karo", 100, "USD", "PKR"),
    ("50 euro to usd kar do", 50, "EUR", "USD"),
    ("ye 200 usd ko pkr me kar do", 200, "USD", "PKR"),
    ("mujhe 500 pkr ko usd me convert karna hai", 500, "PKR", "USD"),
    ("1000 inr ko pkr me badlo", 1000, "INR", "PKR"),
    ("75 gbp kitne usd hain", 75, "GBP", "USD"),
    ("300 aed ko pkr me karo", 300, "AED", "PKR"),
    ("conver 100 usd to pkr", 100, "USD", "PKR"),   # typo
    ("currancy 50 eur to usd", 50, "EUR", "USD"),   # typo
    ("exchang 200 cad to usd", 200, "CAD", "USD"),  # typo
    ("100 dolar to rupees", 100, "USD", "PKR"),
    ("mujhe dollar to rupee chahiye 50 ka", 50, "USD", "PKR"),
    ("convert karo 250 sar to pkr", 250, "SAR", "PKR"),
    ("1000 jpy ko usd mein", 1000, "JPY", "USD"),
    ("bhai 80 usd to inr kar", 80, "USD", "INR"),
    # Spanish/Arabic
    ("convertir 100 USD a EUR", 100, "USD", "EUR"),
    ("كم يساوي 100 دولار بالروبية", 100, "USD", "PKR"),
    ("100 دلار به روپیه", 100, "USD", "PKR"),
]
for prompt, amount, frm, to in currency_adv:
    adversarial.append(ex([user(prompt), assistant(tc({"tool":"currency","args":{"amount":amount,"from":frm,"to":to}}))]))

# ── CONVERT adversarial ──
convert_adv = [
    ("10 km ko miles me karo", 10, "km", "miles"),
    ("5 kg to grams kar do", 5, "kg", "grams"),
    ("isko meters me convert karo — 30 feet", 30, "feet", "meter"),
    ("100 celsius ko fahrenheit mein", 100, "celsius", "fahrenheit"),
    ("conver 10 km to miles", 10, "km", "miles"),   # typo
    ("convrt 500 ml to liter", 500, "ml", "liter"), # typo
    ("1000 grm to kg", 1000, "grams", "kg"),        # typo
    ("kitne miles hain 5 km mein", 5, "km", "miles"),
    ("2 gb ko mb mein batao", 2, "GB", "MB"),
    ("90 minute kitne hours", 90, "minutes", "hours"),
    ("6 feet ko cm me convert karo", 6, "feet", "cm"),
    ("bhai 1 gallon kitne liter hote hain", 1, "gallon", "liter"),
    ("convertir 100 km a millas", 100, "km", "miles"),  # Spanish
    ("تحويل 70 كيلوغرام إلى رطل", 70, "kg", "lbs"),    # Arabic
    ("1 inch ko cm mein", 1, "inch", "cm"),
]
for prompt, val, frm, to in convert_adv:
    adversarial.append(ex([user(prompt), assistant(tc({"tool":"convert","args":{"value":val,"from_unit":frm,"to_unit":to}}))]))

# ── MULTI-TURN adversarial (Hinglish pronoun resolution) ──
multiturn_adv = [
    # "isko" = resolve to prior amount/currency
    [user("100 USD to PKR."),
     assistant(tc({"tool":"currency","args":{"amount":100,"from":"USD","to":"PKR"}})),
     user("isko euros me kar do"),
     assistant(tc({"tool":"currency","args":{"amount":100,"from":"USD","to":"EUR"}}))],

    [user("50 euro to usd"),
     assistant(tc({"tool":"currency","args":{"amount":50,"from":"EUR","to":"USD"}})),
     user("ab isko gbp mein bhi batao"),
     assistant(tc({"tool":"currency","args":{"amount":50,"from":"EUR","to":"GBP"}}))],

    [user("10 km ko miles me batao"),
     assistant(tc({"tool":"convert","args":{"value":10,"from_unit":"km","to_unit":"miles"}})),
     user("ab feet mein bhi"),
     assistant(tc({"tool":"convert","args":{"value":10,"from_unit":"km","to_unit":"feet"}}))],

    [user("lahore ka weather batao"),
     assistant(tc({"tool":"weather","args":{"location":"Lahore","unit":"C"}})),
     user("karachi ka bhi"),
     assistant(tc({"tool":"weather","args":{"location":"Karachi","unit":"C"}}))],

    [user("200 usd ko pkr me karo"),
     assistant(tc({"tool":"currency","args":{"amount":200,"from":"USD","to":"PKR"}})),
     user("same amount aed mein?"),
     assistant(tc({"tool":"currency","args":{"amount":200,"from":"USD","to":"AED"}}))],

    [user("Convert 100 USD to PKR."),
     assistant(tc({"tool":"currency","args":{"amount":100,"from":"USD","to":"PKR"}})),
     user("ye wala EUR mein?"),
     assistant(tc({"tool":"currency","args":{"amount":100,"from":"USD","to":"EUR"}}))],
]
for msgs in multiturn_adv:
    adversarial.append(ex(msgs))

# ── REFUSAL adversarial (shouldn't fire a tool) ──
refusal_adv = [
    ("mujhe pizza order karna hai", "I don't have a food ordering tool. I can help with weather, calendar, conversions, currency, or SQL."),
    ("flight book karo london ke liye", "I don't have a flight booking tool. I can help with weather, calendar, conversions, currency, or SQL."),
    ("gaana chala do", "I don't have a music player tool. I can help with weather, calendar, conversions, currency, or SQL."),
    ("whatsapp message bhejo", "I don't have a messaging tool. I can help with weather, calendar, conversions, currency, or SQL."),
    ("mera bank balance batao", "I don't have a banking tool. I can help with weather, calendar, conversions, currency, or SQL."),
    ("cricket ka score kya hai", "I don't have a sports score tool. I can help with weather, calendar, conversions, currency, or SQL."),
]
for prompt, resp in refusal_adv:
    adversarial.append(ex([user(prompt), assistant(resp)]))

# Append to existing train_data.jsonl
with open("train_data.jsonl","a",encoding="utf-8") as f:
    for e in adversarial:
        f.write(json.dumps(e, ensure_ascii=False)+"\n")

print(f"Added {len(adversarial)} adversarial examples.")

# Verify total
total = sum(1 for _ in open("train_data.jsonl"))
print(f"Total examples now: {total}")

Added 70 adversarial examples.
Total examples now: 567


In [4]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import json

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# System prompt — CRITICAL for refusal behaviour
SYSTEM = (
    "You are Pocket-Agent, an on-device assistant. "
    "You have exactly 5 tools: weather, calendar, convert, currency, sql. "
    "For tool calls emit ONLY: <tool_call>{...}</tool_call> with valid JSON. "
    "If no tool fits — chitchat, impossible tools, ambiguous references with no history — "
    "respond in plain text only. Never emit a tool call when a refusal is correct."
)

def format_example(example):
    messages = [{"role":"system","content":SYSTEM}] + example["messages"]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

# Load dataset
raw = []
with open("train_data.jsonl") as f:
    for line in f: raw.append(json.loads(line))

dataset = Dataset.from_list([{"text": format_example(ex)} for ex in raw])
print(f"Dataset size: {len(dataset)}")

# Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir="./lora_adapter",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
    ),
)
trainer.train()
model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")
print("Training done. Adapter saved.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/567 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 567 | Num Epochs = 3 | Total steps = 108
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 1,081,344 of 495,114,112 (0.22% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,4.064311
20,2.620267
30,1.220734
40,0.525469
50,0.360827
60,0.306221
70,0.290880
80,0.273363
90,0.242266
100,0.230967


Training done. Adapter saved.


In [5]:
# # Merge LoRA into base and save as float16
# from unsloth import FastLanguageModel

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="./lora_adapter",
#     max_seq_length=2048,
#     dtype=None,
#     load_in_4bit=True,
# )

# model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")
# print("Merged model saved.")

# # Export to GGUF Q4_K_M — targets ~310 MB
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF export done.")

# # Check file size
# import os
# gguf_path = [f for f in os.listdir("model_gguf") if f.endswith(".gguf")][0]
# size_mb = os.path.getsize(f"model_gguf/{gguf_path}") / (1024*1024)
# print(f"GGUF file: {gguf_path} — {size_mb:.1f} MB")
# if size_mb <= 250:
#     print("✓ Passes ≤250 MB bonus gate (+10 pts)")
# elif size_mb <= 500:
#     print("✓ Passes ≤500 MB hard gate")
# else:
#     print("✗ FAIL — too large, need to requantize")

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:24<00:00, 24.34s/it]


Unsloth: Merge process complete. Saved to `/content/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/qwen2.5-0.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf_gguf/qwen2.5-0.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model model_gguf_gguf/qwen2.5-0.5b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to model_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f model_gguf_gguf/Modelfile
GGUF export done.


IndexError: list index out of range

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os

bnb_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    "./merged_model",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("./merged_model")

model.save_pretrained("./quantized_model")
tokenizer.save_pretrained("./quantized_model")

size = sum(os.path.getsize(f"./quantized_model/{f}")
           for f in os.listdir("./quantized_model")) / (1024*1024)
print(f"Size: {size:.1f} MB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer you are loading from './merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Size: 462.8 MB


In [30]:
%%writefile inference.py
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, json, re

SYSTEM = """You are Pocket-Agent, an on-device assistant with exactly 5 tools.

TOOL SCHEMAS (emit exactly this JSON structure):
{"tool":"weather",  "args":{"location":"string","unit":"C|F"}}
{"tool":"calendar", "args":{"action":"list|create","date":"YYYY-MM-DD","title":"string (only for create)"}}
{"tool":"convert",  "args":{"value":number,"from_unit":"string","to_unit":"string"}}
{"tool":"currency", "args":{"amount":number,"from":"ISO3","to":"ISO3"}}
{"tool":"sql",      "args":{"query":"string"}}

RULES:
1. Tool call → wrap ONLY in <tool_call>...</tool_call>, nothing else before or after.
2. unit default → always "C" unless user says Fahrenheit/F/°F.
3. ISO codes → USD, PKR, EUR, GBP, AED, INR, JPY, SAR, CAD (3-letter uppercase always).
4. Multi-turn → if user says "that", "it", "same", "isko", "ye wala" — resolve from previous assistant turn.
5. Refusal triggers → chitchat, impossible tools (flight/pizza/music/alarm/translate), ambiguous with no history.
6. NEVER emit a tool call for refusals. Plain text only.
7. NEVER add explanation around a tool call. Just the tag.

EXAMPLES:
User: weather in karachi → <tool_call>{"tool":"weather","args":{"location":"Karachi","unit":"C"}}</tool_call>
User: 100 USD to PKR → <tool_call>{"tool":"currency","args":{"amount":100,"from":"USD","to":"PKR"}}</tool_call>
User: book a flight → I don't have a flight booking tool. I can help with weather, calendar, conversions, currency, or SQL.
User: convert that to EUR [after USD→PKR turn] → <tool_call>{"tool":"currency","args":{"amount":100,"from":"USD","to":"EUR"}}</tool_call>"""

_model = None
_tokenizer = None

def _load():
    global _model, _tokenizer
    if _model is None:
        bnb_config = BitsAndBytesConfig(load_in_4bit=True)
        _tokenizer = AutoTokenizer.from_pretrained("./quantized_model")
        _model = AutoModelForCausalLM.from_pretrained(
            "./quantized_model",
            quantization_config=bnb_config,
            device_map="auto"
        )

# def _extract_tool_call(text: str) -> str:
#     # Truncate anything after </tool_call>
#     if "</tool_call>" in text:
#         text = text[:text.index("</tool_call>") + len("</tool_call>")]

#     match = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
#     if not match:
#         # For plain text refusals, take only first sentence/line
#         return text.split("\n")[0].strip()
#     raw = match.group(1).strip()
#     try:
#         obj = json.loads(raw)
#         if "tool" not in obj or "args" not in obj:
#             return text
#         if obj["tool"] == "currency":
#             obj["args"]["from"] = obj["args"].get("from", "").upper()
#             obj["args"]["to"]   = obj["args"].get("to", "").upper()
#         if obj["tool"] == "weather":
#             if obj["args"].get("unit", "C") not in ("C", "F"):
#                 obj["args"]["unit"] = "C"
#         return f"<tool_call>{json.dumps(obj, separators=(',', ':'))}</tool_call>"
#     except json.JSONDecodeError:
#         return text

def _extract_tool_call(text: str) -> str:
    # Hard stop at known stop tokens
    for stop in ["<|im_end|>", "<|endoftext|>", "\nUser:", "\nuser:", "\nHuman:"]:
        if stop in text:
            text = text[:text.index(stop)]
    text = text.strip()

    # Truncate anything after </tool_call>
    if "</tool_call>" in text:
        text = text[:text.index("</tool_call>") + len("</tool_call>")]

    # If tool_call tag exists anywhere, extract ONLY that — ignore everything else
    match = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    if match:
        raw = match.group(1).strip()
        # Take only up to first closing brace of valid JSON
        try:
            obj = json.loads(raw)
            if "tool" not in obj or "args" not in obj:
                return "I can help with weather, calendar, conversions, currency, or SQL."
            if obj["tool"] == "currency":
                obj["args"]["from"] = obj["args"].get("from", "").upper()
                obj["args"]["to"]   = obj["args"].get("to", "").upper()
            if obj["tool"] == "weather":
                if obj["args"].get("unit", "C") not in ("C", "F"):
                    obj["args"]["unit"] = "C"
            return f"<tool_call>{json.dumps(obj, separators=(',', ':'))}</tool_call>"
        except json.JSONDecodeError:
            # Salvage: find first {...} block
            m2 = re.search(r'(\{.*?\})', text, re.DOTALL)
            if m2:
                try:
                    obj = json.loads(m2.group(1))
                    return f"<tool_call>{json.dumps(obj, separators=(',', ':'))}</tool_call>"
                except:
                    pass
            return "I can help with weather, calendar, conversions, currency, or SQL."

    # # No tool call tag — plain text refusal
    # # Strip non-ASCII garbage, return first clean line
    # clean = re.sub(r'[^\x00-\x7F]+', '', text).strip()
    # if clean:
    #     return clean.split("\n")[0].strip()
    # return "I can help with weather, calendar, conversions, currency, or SQL."

    # No tool call tag — plain text refusal
    clean = re.sub(r'[^\x00-\x7F]+', '', text).strip()  # strip non-ASCII
    clean = re.sub(r'[^\w\s\.\,\!\?\-\'\"]', '', clean).strip()  # strip remaining garbage
    lines = [l.strip() for l in clean.split("\n") if len(l.strip()) > 10]
    if lines:
        return lines[0]
    return "I can help with weather, calendar, conversions, currency, or SQL."


def run(prompt: str, history: list[dict]) -> str:
    _load()
    messages = [{"role": "system", "content": SYSTEM}]
    for turn in history:
        messages.append(turn)
    messages.append({"role": "user", "content": prompt})

    text = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(text, return_tensors="pt").to(_model.device)

    with torch.no_grad():
        outputs = _model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=_tokenizer.eos_token_id,
        )

    with torch.no_grad():
        outputs = _model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=_tokenizer.eos_token_id,
            eos_token_id=_tokenizer.eos_token_id,  # ADD THIS
      )

    response = _tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    return _extract_tool_call(response)

if __name__ == "__main__":
    print("--- Weather ---")
    print(run("What's the weather in Lahore?", []))

    print("--- Fahrenheit ---")
    print(run("Temperature in Dubai in F?", []))

    print("--- Currency ---")
    print(run("Convert 100 USD to PKR.", []))

    print("--- Multi-turn ---")
    history = [
        {"role": "user",      "content": "Convert 100 USD to PKR."},
        {"role": "assistant", "content": '<tool_call>{"tool":"currency","args":{"amount":100,"from":"USD","to":"PKR"}}</tool_call>'},
    ]
    print(run("Now convert that to EUR.", history))

    print("--- Refusal ---")
    print(run("Book me a flight to London.", []))

    print("--- Hinglish ---")
    print(run("lahore ka mausam batao", []))

Overwriting inference.py


In [37]:
%%writefile app.py
import gradio as gr
from inference import run

history = []

def chat(user_msg, chat_history):
    global history
    if not user_msg.strip():
        return "", chat_history
    response = run(user_msg, history)
    display_response = response
    if "<tool_call>" in response:
        display_response = f"🔧 Tool Call:\n```json\n{response.replace('<tool_call>','').replace('</tool_call>','').strip()}\n```"
    history.append({"role": "user", "content": user_msg})
    history.append({"role": "assistant", "content": response})
    chat_history.append({"role": "user", "content": user_msg})
    chat_history.append({"role": "assistant", "content": display_response})
    return "", chat_history

def clear():
    global history
    history = []
    return []

with gr.Blocks(title="Pocket-Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Pocket-Agent\nOn-device tool-calling assistant.")
    chatbot = gr.Chatbot(height=450, type="messages", label="Conversation")
    msg = gr.Textbox(placeholder="e.g. What's the weather in Karachi?", label="Message")
    with gr.Row():
        send = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear", variant="secondary")
    send.click(chat, [msg, chatbot], [msg, chatbot])
    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    clear_btn.click(clear, [], [chatbot])

demo.launch(share=True)

Overwriting app.py


In [38]:
!python inference.py

--- Weather ---
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading weights: 100% 290/290 [00:00<00:00, 1097.00it/s]
<tool_call>{"tool":"weather","args":{"location":"Lahore","unit":"C"}}</tool_call>
--- Fahrenheit ---
<tool_call>{"tool":"weather","args":{"location":"Dubai","unit":"F"}}</tool_call>
--- Currency ---
<tool_call>{"tool":"currency","args":{"amount":100,"from":"USD","to":"PKR"}}</tool_call>
--- Multi-turn ---
<tool_call>{"tool":"currency","args":{"amount":100,"from":"USD","to":"EUR"}}</tool_call>
--- Refusal ---
tt chahi USDEUR turn ota London dict ma ace karaSI
--- Hingli

In [ ]:
!python app.py

/content/app.py:25: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Pocket-Agent", theme=gr.themes.Soft()) as demo:
/content/app.py:27: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=450, type="messages", label="Conversation")
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e8873c8fe1cc317e5d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
